# Step 2 — Load one real SWED satellite tile

This notebook runs in Google Colab. It installs the repository code, optionally downloads the SWED sample into Colab's temporary storage, and verifies that a real image and mask satisfy the same data contract as our synthetic example.

## 1. Clone the code
This downloads only the small source repository—not the satellite dataset.

In [ ]:
REPO_URL = "https://github.com/hriship618/coastline-image-segmentation.git"
!git clone {REPO_URL}
%cd coastline-image-segmentation
%pip install -q -e ".[data]"

## 2. Optional dataset download
The next cell does nothing until `DOWNLOAD_DATA` is changed to `True`. When enabled, it downloads the approximately 1.36 GB SWED sample into `/content/data`, which disappears when the Colab runtime is deleted. Do not commit or redistribute the dataset.

In [ ]:
from pathlib import Path
import subprocess

DOWNLOAD_DATA = False
archive = Path("/content/data/SWED_sample.zip")
data_root = Path("/content/data/swed")
url = "https://ukho-openmldata.s3.eu-west-2.amazonaws.com/SWED_sample.zip"

if DOWNLOAD_DATA:
    archive.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["wget", "-c", url, "-O", str(archive)], check=True)
    data_root.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-q", str(archive), "-d", str(data_root)], check=True)
    print(f"Extracted SWED under {data_root}")
else:
    print("No data downloaded. Set DOWNLOAD_DATA=True when ready.")

## 3. Discover matching image/mask files
SWED uses matching filenames containing `_image_` and `_label_`. Discovery verifies that every image has exactly one label before training can begin.

In [ ]:
from coastlearn.data import FIVE_BANDS, SwedDataset, discover_swed_pairs

pairs = discover_swed_pairs(data_root)
dataset = SwedDataset(pairs, bands=FIVE_BANDS)
example = dataset[0]
print(f"pairs: {len(pairs)}")
print(f"image: {tuple(example['image'].shape)}, {example['image'].dtype}")
print(f"mask:  {tuple(example['mask'].shape)}, {example['mask'].dtype}")

## 4. Visual inspection
The RGB view is made from the red, green, and blue channels. NIR and SWIR are shown separately because human eyes cannot see them. The mask should line up with the visible land/water boundary.

In [ ]:
import matplotlib.pyplot as plt

image = example["image"].numpy()
mask = example["mask"].numpy()
rgb = image[:3].transpose(1, 2, 0)

figure, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(rgb.clip(0, 1)); axes[0].set_title("RGB")
axes[1].imshow(image[3], cmap="gray"); axes[1].set_title("Near infrared")
axes[2].imshow(image[4], cmap="gray"); axes[2].set_title("Shortwave infrared")
axes[3].imshow(mask, vmin=0, vmax=1, cmap="Blues"); axes[3].set_title("Water mask")
for axis in axes: axis.axis("off")
plt.tight_layout()